# Temperature / Top-k / Top-p Sampling from Scratch

Build order:
1. **Stage 1** — Probability distributions & numerical stability (softmax, log-softmax)
2. **Stage 2** — Temperature scaling
3. **Stage 3** — Top-k sampling
4. **Stage 4** — Top-p (nucleus) sampling
5. **Stage 5** — Combined sampling & end-to-end generation

Model: `gpt2` — small enough to run on CPU, large enough to produce meaningful distributions.

## Setup

In [ ]:
!pip install transformers torch matplotlib numpy --quiet

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
MODEL_NAME = "gpt2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device).eval()

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size:   {tokenizer.vocab_size:,}")

---
## Stage 1 — Probability Distributions & Numerical Stability

LLMs output **logits** — raw unnormalized scores for each token in the vocabulary.
To sample the next token, we need to convert these logits into a valid probability distribution.

The standard approach is the **softmax** function:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

However, naive softmax is **numerically unstable** — if logits are large, $e^{z_i}$ can overflow to `inf`.
The fix is to subtract the maximum logit before exponentiating:

$$p_i = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$

This produces identical results (since the constant cancels in the ratio) but keeps values in a numerically safe range.

### 1a — Naive vs. stable softmax

In [ ]:
def naive_softmax(logits: torch.Tensor) -> torch.Tensor:
    """
    Naive softmax: directly exponentiate and normalize.
    This will overflow for large logit values.

    Args:
        logits: [vocab_size] or [batch, vocab_size]

    Returns:
        probs: same shape as logits, sums to 1 along last dim
    """
    # TODO: implement naive softmax
    # Hint: exp(logits) / sum(exp(logits))
    raise NotImplementedError


def stable_softmax(logits: torch.Tensor) -> torch.Tensor:
    """
    Numerically stable softmax: subtract max before exponentiating.

    Args:
        logits: [vocab_size] or [batch, vocab_size]

    Returns:
        probs: same shape as logits, sums to 1 along last dim
    """
    # TODO: implement numerically stable softmax
    # Hint: subtract logits.max(dim=-1, keepdim=True).values before exp
    raise NotImplementedError

In [ ]:
# --- Sanity check: normal-range logits ---
logits_small = torch.tensor([2.0, 1.0, 0.1])
print("Small logits:", logits_small)
print("Naive softmax: ", naive_softmax(logits_small))
print("Stable softmax:", stable_softmax(logits_small))
print("PyTorch F.softmax:", F.softmax(logits_small, dim=-1))
print()

# --- Demonstrate overflow with large logits ---
logits_large = torch.tensor([1000.0, 1001.0, 1002.0])
print("Large logits:", logits_large)
print("Naive softmax: ", naive_softmax(logits_large), "  ← contains nan from inf/inf")
print("Stable softmax:", stable_softmax(logits_large), "  ← correct")
print("PyTorch F.softmax:", F.softmax(logits_large, dim=-1))

### 1b — Log-softmax and why it matters

For training (cross-entropy loss) and for sampling via the Gumbel-max trick, we often need **log-probabilities** rather than probabilities.

Computing `log(softmax(z))` naively suffers from two problems:
1. Softmax can underflow to 0 for tokens with low logits, making `log(0) = -inf`
2. We're doing unnecessary work: `exp()` then `log()` mostly cancel

The numerically stable **log-softmax** avoids both issues:

$$\log \text{softmax}(z_i) = z_i - \max(z) - \log\sum_j e^{z_j - \max(z)}$$

In [ ]:
def stable_log_softmax(logits: torch.Tensor) -> torch.Tensor:
    """
    Numerically stable log-softmax.

    log(softmax(z_i)) = (z_i - max(z)) - log(sum_j exp(z_j - max(z)))

    Args:
        logits: [vocab_size] or [batch, vocab_size]

    Returns:
        log_probs: same shape, all values <= 0, logsumexp = 0
    """
    # TODO: implement numerically stable log-softmax
    # Hint: first compute shifted = logits - max(logits)
    # Then log_sum_exp = log(sum(exp(shifted)))
    # Return shifted - log_sum_exp
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
logits_test = torch.randn(5)
print("Logits:", logits_test)
print()

log_probs_ours = stable_log_softmax(logits_test)
log_probs_ref  = F.log_softmax(logits_test, dim=-1)
print("Our log-softmax:    ", log_probs_ours)
print("PyTorch log-softmax:", log_probs_ref)
print("Match:", torch.allclose(log_probs_ours, log_probs_ref, atol=1e-6))
print()

# Verify: exp(log_softmax) recovers softmax
probs_from_log = torch.exp(log_probs_ours)
probs_direct   = stable_softmax(logits_test)
print("exp(log_softmax):", probs_from_log)
print("softmax:         ", probs_direct)
print("Match:", torch.allclose(probs_from_log, probs_direct, atol=1e-6))

### 1c — Visualizing model logit distributions

Let's look at what real logit distributions from GPT-2 look like and why numerical stability matters in practice.

In [ ]:
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits[0, -1, :]  # [vocab_size] — last token position

print(f"Logit shape: {logits.shape}")
print(f"Logit range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
print(f"Logit mean:  {logits.mean().item():.2f}")
print(f"Logit std:   {logits.std().item():.2f}")

probs = F.softmax(logits, dim=-1)
top_probs, top_indices = probs.topk(10)
print(f"\nTop-10 tokens and their probabilities:")
for i in range(10):
    token = tokenizer.decode([top_indices[i].item()])
    print(f"  {token!r:>15s}  p={top_probs[i].item():.4f}")

print(f"\nSum of top-10 probs: {top_probs.sum().item():.4f}")
print(f"Sum of all probs:    {probs.sum().item():.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Logit histogram
axes[0].hist(logits.cpu().numpy(), bins=100, color="steelblue", alpha=0.7)
axes[0].set_title("Raw Logit Distribution")
axes[0].set_xlabel("Logit value")
axes[0].set_ylabel("Count")
axes[0].grid(True, alpha=0.3)

# Plot 2: Probability histogram (note the extreme skew)
axes[1].hist(probs.cpu().numpy(), bins=100, color="coral", alpha=0.7)
axes[1].set_title("Probability Distribution (softmax)")
axes[1].set_xlabel("Probability")
axes[1].set_ylabel("Count")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

# Plot 3: Top-50 probabilities (sorted)
top50_probs, top50_idx = probs.topk(50)
axes[2].bar(range(50), top50_probs.cpu().numpy(), color="seagreen", alpha=0.7)
axes[2].set_title("Top-50 Token Probabilities")
axes[2].set_xlabel("Rank")
axes[2].set_ylabel("Probability")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Stage 2 — Temperature Scaling

Temperature is the simplest way to control the "sharpness" of a probability distribution.
Before applying softmax, we divide the logits by a temperature parameter $T > 0$:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **T = 1.0**: Standard softmax (no change)
- **T → 0**: Distribution becomes more peaked (greedy / deterministic)
- **T → ∞**: Distribution becomes more uniform (random)

**Intuition:** Dividing by a large T "compresses" the logit differences, making all tokens more equally likely. Dividing by a small T "amplifies" the differences, making the model more confident in its top choices.

### 2a — Implement temperature-scaled softmax

In [ ]:
def temperature_softmax(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """
    Apply temperature scaling then softmax.

    Args:
        logits:      [vocab_size] raw model logits
        temperature: positive float. 1.0 = standard softmax.
                     <1.0 = sharper, >1.0 = flatter.

    Returns:
        probs: [vocab_size] valid probability distribution
    """
    if temperature <= 0:
        raise ValueError("Temperature must be positive")

    # TODO: divide logits by temperature, then apply softmax
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
test_logits = torch.tensor([3.0, 1.0, 0.5, -1.0, -2.0])

for T in [0.1, 0.5, 1.0, 2.0, 10.0]:
    p = temperature_softmax(test_logits, T)
    print(f"T={T:>4.1f}  probs={p.numpy().round(4)}  sum={p.sum().item():.4f}  max={p.max().item():.4f}  entropy={-(p * p.log()).sum().item():.4f}")

### 2b — Visualize temperature effects on real model outputs

In [ ]:
temperatures = [0.1, 0.5, 1.0, 1.5, 2.0, 5.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, T in enumerate(temperatures):
    probs_t = temperature_softmax(logits, T)
    top_k_probs, top_k_idx = probs_t.topk(20)
    tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_k_idx]

    axes[i].barh(range(20), top_k_probs.cpu().numpy(), color="steelblue", alpha=0.7)
    axes[i].set_yticks(range(20))
    axes[i].set_yticklabels(tokens, fontsize=7)
    axes[i].invert_yaxis()
    axes[i].set_title(f"T = {T}")
    axes[i].set_xlabel("Probability")

    entropy = -(probs_t * (probs_t + 1e-10).log()).sum().item()
    axes[i].text(0.95, 0.95, f"H={entropy:.2f}", transform=axes[i].transAxes,
                 ha="right", va="top", fontsize=9,
                 bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle(f'Temperature scaling — prompt: "{prompt}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2c — Sample tokens with temperature

Now let's use temperature to actually sample next tokens.

In [ ]:
def sample_with_temperature(
    logits: torch.Tensor,
    temperature: float = 1.0,
    num_samples: int = 1,
) -> torch.Tensor:
    """
    Sample token(s) from logits with temperature scaling.

    Args:
        logits:      [vocab_size]
        temperature: >0, controls randomness
        num_samples: how many tokens to sample

    Returns:
        token_ids: [num_samples]
    """
    # TODO: use temperature_softmax to get probs, then torch.multinomial to sample
    raise NotImplementedError

In [ ]:
# --- Sanity check: sample 20 tokens at different temperatures ---
print(f'Prompt: "{prompt}"\n')

for T in [0.1, 0.5, 1.0, 2.0]:
    samples = sample_with_temperature(logits, temperature=T, num_samples=20)
    decoded = [tokenizer.decode([s.item()]).strip() for s in samples]
    unique_tokens = len(set(decoded))
    print(f"T={T:>3.1f} ({unique_tokens:>2d} unique): {decoded}")

---
## Stage 3 — Top-k Sampling

Even with low temperature, the model can still sample from unlikely tokens because softmax assigns non-zero probability to **every** token. Top-k sampling truncates the distribution to only the **k most probable** tokens, then renormalizes.

```
logits → keep only top-k → renormalize → sample
```

**Problem with top-k:** The right value of k is context-dependent. For a highly confident prediction (e.g., closing a parenthesis), k=5 might include garbage tokens. For an open-ended continuation, k=50 might be too restrictive.

### 3a — Implement top-k filtering

In [ ]:
def top_k_filtering(logits: torch.Tensor, k: int) -> torch.Tensor:
    """
    Zero out all logits except the top-k highest, by setting
    the filtered positions to -inf (so softmax gives them 0 probability).

    Args:
        logits: [vocab_size] raw logits
        k:      number of top tokens to keep

    Returns:
        filtered_logits: [vocab_size] with non-top-k set to -inf
    """
    if k <= 0:
        raise ValueError("k must be positive")
    if k >= logits.shape[-1]:
        return logits

    # TODO: find the k-th largest logit value as a threshold
    # Hint: use logits.topk(k) to get the top-k values
    # The k-th largest is top_k_values[..., -1]
    raise NotImplementedError

    # TODO: clone logits and set everything below the threshold to -inf
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
test_logits = torch.tensor([5.0, 3.0, 2.0, 1.0, 0.5, -1.0, -3.0])

for k in [1, 3, 5, 7]:
    filtered = top_k_filtering(test_logits, k)
    probs = F.softmax(filtered, dim=-1)
    print(f"k={k}  filtered_logits={filtered.numpy()}")
    print(f"     probs={probs.numpy().round(4)}  sum={probs.sum().item():.4f}")
    print()

### 3b — Sample with top-k

In [ ]:
def sample_top_k(
    logits: torch.Tensor,
    k: int,
    temperature: float = 1.0,
    num_samples: int = 1,
) -> torch.Tensor:
    """
    Sample from top-k filtered distribution with temperature.

    Pipeline: logits → temperature scale → top-k filter → softmax → sample

    Args:
        logits:      [vocab_size]
        k:           number of top tokens to keep
        temperature: temperature scaling (applied before filtering)
        num_samples: how many tokens to draw

    Returns:
        token_ids: [num_samples]
    """
    # TODO: apply temperature, then top-k filtering, then softmax, then multinomial
    raise NotImplementedError

In [ ]:
# --- Sanity check: compare top-k with different k values ---
print(f'Prompt: "{prompt}"\n')

for k in [1, 5, 10, 50, 200]:
    samples = sample_top_k(logits, k=k, temperature=1.0, num_samples=20)
    decoded = [tokenizer.decode([s.item()]).strip() for s in samples]
    unique_tokens = len(set(decoded))
    print(f"k={k:>3d} ({unique_tokens:>2d} unique): {decoded}")

### 3c — Visualize top-k effect

In [ ]:
k_values = [5, 10, 50, 200]

fig, axes = plt.subplots(1, len(k_values), figsize=(18, 4))

for i, k in enumerate(k_values):
    filtered = top_k_filtering(logits, k)
    probs_k = F.softmax(filtered, dim=-1)

    top_probs, top_idx = probs_k.topk(min(k, 20))
    tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_idx]

    axes[i].barh(range(len(tokens)), top_probs.cpu().numpy(), color="steelblue", alpha=0.7)
    axes[i].set_yticks(range(len(tokens)))
    axes[i].set_yticklabels(tokens, fontsize=7)
    axes[i].invert_yaxis()
    axes[i].set_title(f"Top-k = {k}")
    axes[i].set_xlabel("Probability")

plt.suptitle(f'Top-k filtering — prompt: "{prompt}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Stage 4 — Top-p (Nucleus) Sampling

Top-p sampling (Holtzman et al., 2020) solves the problem of fixed k by dynamically choosing how many tokens to keep. Instead of a fixed count, we keep the **smallest set** of tokens whose cumulative probability exceeds a threshold p:

$$\text{Top-p}(z) = \{i : i \in \text{sorted tokens by decreasing probability, up to cumulative sum} \geq p\}$$

```
logits → softmax → sort by probability → compute cumulative sum → keep tokens until sum ≥ p → renormalize → sample
```

- **p = 0.1**: Very narrow (only the most likely tokens)
- **p = 0.9**: Wide (most of the probability mass, common default)
- **p = 1.0**: No filtering (equivalent to standard sampling)

### 4a — Implement top-p filtering

In [ ]:
def top_p_filtering(logits: torch.Tensor, p: float) -> torch.Tensor:
    """
    Nucleus sampling: keep the smallest set of tokens whose cumulative
    probability exceeds p, setting everything else to -inf.

    Args:
        logits: [vocab_size] raw logits
        p:      probability threshold in (0, 1]

    Returns:
        filtered_logits: [vocab_size] with low-probability tokens set to -inf

    Steps:
      1. Sort logits in descending order
      2. Compute softmax on sorted logits to get sorted probs
      3. Compute cumulative sum of sorted probs
      4. Create mask: positions where (cumsum - current_prob) >= p should be removed
      5. Set masked positions to -inf in sorted logits
      6. Scatter back to original order
    """
    if p <= 0 or p > 1:
        raise ValueError("p must be in (0, 1]")
    if p == 1.0:
        return logits

    # TODO: sort logits descending, compute sorted probs and cumulative sum
    raise NotImplementedError

    # TODO: create mask for tokens to remove (where cumsum - prob >= p)
    # This ensures the token that pushes past p is kept
    raise NotImplementedError

    # TODO: set masked positions to -inf, scatter back to original order
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
test_logits = torch.tensor([5.0, 3.0, 2.0, 1.0, 0.5, -1.0, -3.0])
test_probs = F.softmax(test_logits, dim=-1)
print("Original probs:", test_probs.numpy().round(4))
print("Cumulative sum: ", test_probs.sort(descending=True).values.cumsum(dim=-1).numpy().round(4))
print()

for p_val in [0.5, 0.8, 0.9, 0.95, 1.0]:
    filtered = top_p_filtering(test_logits, p_val)
    probs = F.softmax(filtered, dim=-1)
    n_kept = (probs > 0).sum().item()
    print(f"p={p_val:.2f}  kept={n_kept}  probs={probs.numpy().round(4)}  sum={probs.sum().item():.4f}")

### 4b — Sample with top-p

In [ ]:
def sample_top_p(
    logits: torch.Tensor,
    p: float,
    temperature: float = 1.0,
    num_samples: int = 1,
) -> torch.Tensor:
    """
    Sample from top-p filtered distribution with temperature.

    Pipeline: logits → temperature scale → top-p filter → softmax → sample

    Args:
        logits:      [vocab_size]
        p:           nucleus probability threshold
        temperature: temperature scaling (applied before filtering)
        num_samples: how many tokens to draw

    Returns:
        token_ids: [num_samples]
    """
    # TODO: apply temperature, then top-p filtering, then softmax, then multinomial
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
print(f'Prompt: "{prompt}"\n')

for p_val in [0.1, 0.5, 0.9, 0.95]:
    # Count how many tokens survive the filter
    filtered = top_p_filtering(logits, p_val)
    n_kept = (filtered > float("-inf")).sum().item()

    samples = sample_top_p(logits, p=p_val, temperature=1.0, num_samples=20)
    decoded = [tokenizer.decode([s.item()]).strip() for s in samples]
    unique_tokens = len(set(decoded))
    print(f"p={p_val:.2f} (kept {n_kept:>5d} tokens, {unique_tokens:>2d} unique in sample): {decoded}")

### 4c — Visualize top-p: dynamic vocabulary size

In [ ]:
p_values = [0.3, 0.5, 0.8, 0.95]

fig, axes = plt.subplots(1, len(p_values), figsize=(18, 4))

for i, p_val in enumerate(p_values):
    filtered = top_p_filtering(logits, p_val)
    probs_p = F.softmax(filtered, dim=-1)
    n_kept = (probs_p > 0).sum().item()

    show_n = min(n_kept, 20)
    top_probs, top_idx = probs_p.topk(show_n)
    tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_idx]

    axes[i].barh(range(show_n), top_probs.cpu().numpy(), color="coral", alpha=0.7)
    axes[i].set_yticks(range(show_n))
    axes[i].set_yticklabels(tokens, fontsize=7)
    axes[i].invert_yaxis()
    axes[i].set_title(f"Top-p = {p_val}  ({n_kept} tokens kept)")
    axes[i].set_xlabel("Probability")

plt.suptitle(f'Top-p (nucleus) filtering — prompt: "{prompt}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 4d — Compare top-k vs top-p: adaptive behavior

The key advantage of top-p is that it adapts to the shape of the distribution.
When the model is very confident, top-p keeps fewer tokens.
When the model is uncertain, top-p keeps more tokens.

In [ ]:
# Get logits for two different contexts: one confident, one uncertain
prompts = [
    "The capital of France is",
    "I think the best way to",
    "def fibonacci(n):\n    return",
    "Once upon a time in a",
]

p_threshold = 0.9
k_fixed = 50

print(f"{'Prompt':<35s} | {'Top-k=50':>10s} | {'Top-p=0.9':>10s} | {'Top-1 prob':>10s}")
print("-" * 80)

for prompt_text in prompts:
    ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(ids)
        lgt = out.logits[0, -1, :]

    # Top-p: count kept tokens
    filtered_p = top_p_filtering(lgt, p_threshold)
    n_kept_p = (filtered_p > float("-inf")).sum().item()

    # Top-1 probability
    top1_prob = F.softmax(lgt, dim=-1).max().item()

    print(f"{prompt_text:<35s} | {k_fixed:>10d} | {n_kept_p:>10d} | {top1_prob:>10.4f}")

---
## Stage 5 — Combined Sampling & End-to-End Generation

In practice, these strategies are often combined:

```
logits → temperature scaling → top-k filter → top-p filter → softmax → sample
```

Applying temperature first adjusts the overall distribution shape, then top-k and top-p prune it to a reasonable subset.

### 5a — Combined sampling function

In [ ]:
def sample_next_token(
    logits: torch.Tensor,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
) -> torch.Tensor:
    """
    Sample a single next token using combined temperature + top-k + top-p.

    Args:
        logits:      [vocab_size] raw model logits
        temperature: temperature scaling (0 = greedy/argmax)
        top_k:       if >0, keep only top-k tokens
        top_p:       if <1.0, apply nucleus filtering

    Returns:
        token_id: scalar tensor
    """
    # TODO: handle greedy case (temperature == 0) by returning argmax
    raise NotImplementedError

    # TODO: apply temperature, then top-k (if >0), then top-p (if <1),
    # then softmax, then multinomial sampling
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
print(f'Prompt: "{prompt}"\n')

configs = [
    {"temperature": 0,   "top_k": 0,  "top_p": 1.0, "label": "greedy"},
    {"temperature": 1.0, "top_k": 50, "top_p": 1.0, "label": "T=1.0, k=50"},
    {"temperature": 1.0, "top_k": 0,  "top_p": 0.9, "label": "T=1.0, p=0.9"},
    {"temperature": 0.8, "top_k": 50, "top_p": 0.9, "label": "T=0.8, k=50, p=0.9"},
]

for cfg in configs:
    tokens = []
    for _ in range(15):
        t = sample_next_token(logits, temperature=cfg["temperature"], top_k=cfg["top_k"], top_p=cfg["top_p"])
        tokens.append(tokenizer.decode([t.item()]).strip())
    unique = len(set(tokens))
    print(f"{cfg['label']:>25s} ({unique:>2d} unique): {tokens}")

### 5b — Autoregressive generation loop

In [ ]:
def generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
) -> str:
    """
    Autoregressive text generation with KV cache.

    Args:
        model:          causal LM
        tokenizer:      associated tokenizer
        prompt:         input text
        max_new_tokens: number of tokens to generate
        temperature:    temperature parameter (0 = greedy)
        top_k:          if >0, apply top-k filtering
        top_p:          if <1.0, apply nucleus filtering

    Returns:
        Full decoded string (prompt + generation)

    Steps:
      1. Encode prompt to input_ids
      2. In a loop:
         a. Run model forward pass (with KV cache)
         b. Extract logits at last position
         c. Use sample_next_token to pick the next token
         d. Append token and continue
      3. Stop on eos_token or max_new_tokens
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids.clone()
    past_key_values = None

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # TODO: run model forward pass with KV cache
            # On first step, feed full generated; after that, feed only last token
            raise NotImplementedError

            # TODO: extract logits at last position
            raise NotImplementedError

            # TODO: sample next token using sample_next_token
            raise NotImplementedError

            # TODO: append to generated, break on eos
            raise NotImplementedError

    return tokenizer.decode(generated[0], skip_special_tokens=True)

In [ ]:
# --- Generate with different sampling strategies ---
gen_prompt = "The meaning of life is"

strategies = [
    {"temperature": 0,   "top_k": 0,  "top_p": 1.0, "label": "Greedy (T=0)"},
    {"temperature": 0.7, "top_k": 0,  "top_p": 1.0, "label": "T=0.7"},
    {"temperature": 1.0, "top_k": 0,  "top_p": 1.0, "label": "T=1.0 (default)"},
    {"temperature": 1.5, "top_k": 0,  "top_p": 1.0, "label": "T=1.5 (creative)"},
    {"temperature": 1.0, "top_k": 50, "top_p": 1.0, "label": "Top-k=50"},
    {"temperature": 1.0, "top_k": 0,  "top_p": 0.9, "label": "Top-p=0.9"},
    {"temperature": 0.8, "top_k": 50, "top_p": 0.95, "label": "T=0.8, k=50, p=0.95"},
]

print(f'Prompt: "{gen_prompt}"')
print("=" * 100)

for strat in strategies:
    text = generate(
        model, tokenizer, gen_prompt,
        max_new_tokens=60,
        temperature=strat["temperature"],
        top_k=strat["top_k"],
        top_p=strat["top_p"],
    )
    print(f"\n[{strat['label']}]")
    print(text)
    print("-" * 100)

### 5c — Entropy and diversity analysis

Let's measure how different strategies affect the **entropy** (uncertainty) and **diversity** (unique tokens) of generated text.

In [ ]:
def measure_generation_stats(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
) -> dict:
    """
    Generate text and collect statistics about the sampling distributions
    encountered at each step.

    Returns dict with:
        - text: generated text
        - entropies: list of Shannon entropies per step
        - unique_ratio: fraction of unique tokens in the output

    Shannon entropy: H = -sum(p * log(p))
    Higher entropy = more uncertainty in the sampling distribution.
    """
    # TODO: implement generation loop that also records entropy at each step
    # Hint: at each step, compute the final sampling distribution (after
    # temperature + top-k + top-p), then compute entropy = -(p * log(p)).sum()
    # Also track unique_ratio = len(unique tokens) / total tokens
    raise NotImplementedError

In [ ]:
analysis_strategies = [
    {"temperature": 0.3, "top_k": 0,  "top_p": 1.0,  "label": "T=0.3",              "color": "navy"},
    {"temperature": 1.0, "top_k": 0,  "top_p": 1.0,  "label": "T=1.0",              "color": "steelblue"},
    {"temperature": 1.0, "top_k": 50, "top_p": 1.0,  "label": "T=1.0, k=50",        "color": "seagreen"},
    {"temperature": 1.0, "top_k": 0,  "top_p": 0.9,  "label": "T=1.0, p=0.9",       "color": "coral"},
    {"temperature": 0.8, "top_k": 50, "top_p": 0.95, "label": "T=0.8, k=50, p=0.95", "color": "purple"},
]

results = []
for strat in analysis_strategies:
    stats = measure_generation_stats(
        model, tokenizer, gen_prompt,
        max_new_tokens=100,
        temperature=strat["temperature"],
        top_k=strat["top_k"],
        top_p=strat["top_p"],
    )
    stats["label"] = strat["label"]
    stats["color"] = strat["color"]
    results.append(stats)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Entropy over time
for r in results:
    axes[0].plot(r["entropies"], label=r["label"], color=r["color"], alpha=0.8)
axes[0].set_xlabel("Generation Step")
axes[0].set_ylabel("Shannon Entropy (nats)")
axes[0].set_title("Sampling Distribution Entropy per Step")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Mean entropy bar chart
labels = [r["label"] for r in results]
mean_entropies = [np.mean(r["entropies"]) for r in results]
colors = [r["color"] for r in results]
axes[1].barh(labels, mean_entropies, color=colors, alpha=0.7)
axes[1].set_xlabel("Mean Entropy (nats)")
axes[1].set_title("Average Entropy by Strategy")
axes[1].grid(True, alpha=0.3, axis="x")

# Plot 3: Unique token ratio
unique_ratios = [r["unique_ratio"] for r in results]
axes[2].barh(labels, unique_ratios, color=colors, alpha=0.7)
axes[2].set_xlabel("Unique Token Ratio")
axes[2].set_title("Lexical Diversity")
axes[2].set_xlim(0, 1)
axes[2].grid(True, alpha=0.3, axis="x")

plt.suptitle(f'Generation analysis — prompt: "{gen_prompt}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'Strategy':<30s} | {'Mean H':>8s} | {'Unique %':>8s} | {'Tokens':>6s}")
print("-" * 65)
for r in results:
    print(f"{r['label']:<30s} | {np.mean(r['entropies']):>8.2f} | {r['unique_ratio']:>7.1%} | {r['n_tokens']:>6d}")